# DQ 检查 — ODS 层（Rule 0a/0b）

在 DWD 之前运行两类检查：
- **Rule 0a**（block）：每个来源的 ODS 分区行数 > 0 — 空分区/文件未到必须阻断
- **Rule 0b**（warn）：与上周同日相比偏差 < 30% — 超阈值记录日志告警，不阻断

只有 Rule 0a FAIL 才抛出异常阻断 DWD；Rule 0b 仅告警，允许节假日/周末正常波动通过。

## 1. SparkSession

In [1]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterODS")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE    = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y   = f"{BASE}/data/warehouse/ods/yellow"
ODS_G   = f"{BASE}/data/warehouse/ods/green"
ODS_FHV = f"{BASE}/data/warehouse/ods/fhv_hv"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


26/10/19 14:55:35 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/10/19 14:55:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 14:55:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TARGET_DATE=2024-03-15
Spark 3.5.1 ready | TARGET_DATE=2024-03-15


## 2. DQ 框架

In [ ]:
results = []

def check(rule_id, layer, rule_type, description, passed, detail="", severity="block"):
    status = "PASS" if passed else "FAIL"
    results.append({"rule_id": rule_id, "status": status, "description": description, "severity": severity})
    mark  = "✅" if passed else ("⚠️ " if severity == "warn" else "❌")
    label = " [WARN]" if (not passed and severity == "warn") else ""
    print(f"{mark} Rule {rule_id} [{layer}]{label} {description}  {detail}")

## 3. Rule 0a：各来源分区行数非空

In [ ]:
from datetime import date, timedelta

sources = [
    ("yellow", ODS_Y),
    ("green",  ODS_G),
    ("fhv_hv", ODS_FHV),
]

target_dt = date(yr, mo, day)

for src, path in sources:
    df_all = spark.read.parquet(path)
    daily_counts = {
        (r.pickup_year, r.pickup_month, r.pickup_day): r["count"]
        for r in df_all.groupBy("pickup_year", "pickup_month", "pickup_day")
                       .count().collect()
    }

    today_count = daily_counts.get((yr, mo, day), 0)

    # Rule 0a (block): partition must be non-empty
    check(f"0a-{src}", "ODS", "non_empty",
          f"{src} partition non-empty for {TARGET_DATE}",
          passed=(today_count > 0),
          detail=f"today={today_count:,}  days_in_ODS={len(daily_counts)}",
          severity="block")

    # Rule 0b (warn): compare to same weekday last week
    # Rolling 7-day avg misclassifies weekends/holidays as anomalies; same-weekday is a fairer baseline
    same_day_last_week = target_dt - timedelta(days=7)
    key_lw = (same_day_last_week.year, same_day_last_week.month, same_day_last_week.day)
    if key_lw in daily_counts:
        lw_count  = daily_counts[key_lw]
        deviation = abs(today_count - lw_count) / lw_count if lw_count > 0 else 0
        check(f"0b-{src}", "ODS", "row_volatility",
              f"{src} count vs same weekday last week ≤30%",
              passed=(deviation < 0.30),
              detail=f"today={today_count:,}  last_week={lw_count:,}  deviation={deviation:.1%}",
              severity="warn")
    else:
        print(f"⚠️  {src}: no data for {same_day_last_week} (first week) — Rule 0b skipped")

## 4. Rule 0b：月度行数波动检测（偏差 < 50%）

In [ ]:
# Rule 0a and 0b are now fully handled in the cell above (c07).
# Each source's daily count is checked against the rolling 7-day average in one loop.
# This cell is intentionally left as a no-op to preserve notebook cell numbering.
print("Rules 0a/0b checks complete (handled in previous cell).")


## 5. 报告 + 阻断判断

In [ ]:
# Persist DQ results before spark.stop() and before any raise
# gate is a partition key so gate1_ods and gate2_dwd don't overwrite each other
DQ_RESULTS_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/warehouse/dq/dq_results")
if results:
    from pyspark.sql import Row
    rows = [Row(
        check_date=TARGET_DATE,
        rule_id=str(r["rule_id"]),
        layer=r.get("layer", ""),
        rule_type=r.get("rule_type", ""),
        description=r["description"],
        status=r["status"],
        severity=r.get("severity", "block"),
        bad_count=int(r.get("bad_count", 0) or 0),
        total_count=int(r.get("total", 0) or 0),
        threshold=float(r["threshold"]) if r.get("threshold") is not None else None,
        gate="gate1_ods",
        pickup_year=yr,
        pickup_month=mo,
        pickup_day=day,
    ) for r in results]
    df = spark.createDataFrame(rows)
    df = df.withColumn("bad_rate",
        F.when(F.col("total_count") > 0,
               F.round(F.col("bad_count") / F.col("total_count"), 6))
         .otherwise(F.lit(None)))
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    (df.write
       .partitionBy("gate", "pickup_year", "pickup_month", "pickup_day")
       .mode("overwrite")
       .parquet(DQ_RESULTS_PATH))
    print(f"DQ results persisted: {len(rows)} rules → {DQ_RESULTS_PATH}")

In [ ]:
block_fails = [r for r in results if r['status'] == 'FAIL' and r.get('severity', 'block') == 'block']
warn_fails  = [r for r in results if r['status'] == 'FAIL' and r.get('severity', 'block') == 'warn']
pass_rules  = [r for r in results if r['status'] == 'PASS']

print(f"\n{'='*50}")
print(f"DQ Report (ODS)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(block_fails)} BLOCK-FAIL | {len(warn_fails)} WARN")
print(f"={'='*49}")

spark.stop()

if warn_fails:
    for r in warn_fails:
        print(f"  ⚠️  {r['rule_id']}: {r['description']}")

if block_fails:
    for r in block_fails:
        print(f"  ❌ {r['rule_id']}: {r['description']}")
    raise ValueError(
        f"DQ FAILED: {len(block_fails)} blocking ODS rule(s). "
        "Pipeline halted — check source data before DWD build."
    )
else:
    print("No blocking ODS failures — proceeding to DWD.")

print("DQ after-ODS check complete.")